# 🏆 EPL SCOUT REPORT 2024-25
## Complete Data Analysis - Player Identification & Strategic Insights

**Angle Métier**: Identify elite players, undervalued gems, and strategic insights for recruitment decisions.

---

### 📋 Deliverables
- Elite player identification
- Undervalued player discovery
- Position-specific analysis
- Club performance ranking
- Interactive Plotly visualizations
- Power BI-ready CSV exports

In [ ]:
# ============================================================================
# IMPORTS & SETUP
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plotly for interactive visuals
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Statistics & Clustering
from scipy.stats import pearsonr, spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Styling
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

import os
os.makedirs('exports', exist_ok=True)
os.makedirs('visualizations', exist_ok=True)

print("✓ All libraries imported successfully")

## [1] DATA LOADING & PROFILING

In [ ]:
# Load dataset (adjust path)
df = pd.read_csv('epl_player_stats_24_25.csv')  # Change path if needed

print(f"Dataset Shape: {df.shape}")
print(f"\nColumns ({len(df.columns)}): {df.columns.tolist()}")

# Display sample
display(df.head(3))
print("\nData Types & Missing Values:")
display(df.info())

## [2] DATA CLEANING

In [ ]:
df = df.copy()

# Clean column names
df.columns = df.columns.str.strip()

# Convert numeric columns
numeric_cols = ['Goals', 'Assists', 'Minutes', 'Appearances', 'Shots', 
                'Shots On Target', 'Yellow Cards', 'Red Cards', 'Saves', 'xG']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"Missing values BEFORE cleaning:")
print(df.isnull().sum())

# Smart NaN handling
df[['Goals', 'Assists']] = df[['Goals', 'Assists']].fillna(0)
df = df.dropna(subset=['Minutes', 'Appearances'])
df[numeric_cols] = df[numeric_cols].fillna(0)

# Remove players with 0 appearances
df = df[df['Appearances'] > 0]

print(f"\n✓ Data cleaned. Remaining players: {len(df)}")

## [3] FEATURE ENGINEERING

In [ ]:
# Per-90-minute normalization (football standard)
df['Goals_per90'] = np.where(df['Minutes'] > 0, (df['Goals'] / df['Minutes']) * 90, 0)
df['Assists_per90'] = np.where(df['Minutes'] > 0, (df['Assists'] / df['Minutes']) * 90, 0)
df['Shots_per90'] = np.where(df['Minutes'] > 0, (df['Shots'] / df['Minutes']) * 90, 0)

# Shot accuracy & conversion
df['Shot_Accuracy'] = np.where(df['Shots'] > 0, 
                               (df['Shots On Target'] / df['Shots']) * 100, 0)
df['Conversion_Rate'] = np.where(df['Shots'] > 0, 
                                  (df['Goals'] / df['Shots']) * 100, 0)

# Discipline metrics
df['Total_Cards'] = df['Yellow Cards'] + (df['Red Cards'] * 2)
df['Cards_per90'] = np.where(df['Minutes'] > 0, 
                              (df['Total_Cards'] / df['Minutes']) * 90, 0)

# PERFORMANCE SCORE (Composite Metric)
df['Performance_Score'] = (
    (df['Goals_per90'] * 2) +           # Goals weighted 2x
    (df['Assists_per90'] * 1.5) +       # Assists weighted 1.5x
    (df['Shot_Accuracy'] * 0.1)         # Accuracy bonus
)

# Categorizations
df['Playing_Time_Category'] = pd.cut(df['Minutes'], 
                                     bins=[0, 450, 1350, 2700, 5000],
                                     labels=['Reserve', 'Squad Rotation', 'Regular', 'Key Player'])

df['Age_Group'] = pd.cut(df['Age'], 
                         bins=[15, 22, 28, 32, 40],
                         labels=['Young (16-22)', 'Prime (23-28)', 'Experienced (29-32)', 'Veteran (33+)'])

print("✓ Features engineered:")
print("  - Per-90 normalization")
print("  - Shot efficiency metrics")
print("  - Discipline analysis")
print("  - Performance Score (composite)")
print("  - Age & Playing Time categories\n")

display(df[['Player Name', 'Club', 'Goals_per90', 'Assists_per90', 
          'Performance_Score', 'Playing_Time_Category']].head(10))

## [4] PLAYER SEGMENTATION & CLASSIFICATION

In [ ]:
MIN_MINUTES = 450  # ~5 full matches
df_filtered = df[df['Minutes'] >= MIN_MINUTES].copy()

print(f"Players with {MIN_MINUTES}+ minutes: {len(df_filtered)}\n")

# Elite tier classification
perf_q1 = df_filtered['Performance_Score'].quantile(0.75)
perf_q2 = df_filtered['Performance_Score'].quantile(0.50)
perf_q3 = df_filtered['Performance_Score'].quantile(0.25)

df_filtered['Elite_Tier'] = pd.cut(df_filtered['Performance_Score'],
                                   bins=[0, perf_q3, perf_q2, perf_q1, 1000],
                                   labels=['Below Average', 'Average', 'Above Average', 'Elite'])

# Undervalued players
df_filtered['Is_Undervalued'] = (
    (df_filtered['Performance_Score'] > df_filtered['Performance_Score'].median()) &
    (df_filtered['Minutes'] < 1350)
)

# Overperformers vs xG
if 'xG' in df.columns and df['xG'].sum() > 0:
    df_filtered['Goals_vs_xG'] = df_filtered['Goals'] - df_filtered['xG']
    df_filtered['Is_Overperformer'] = df_filtered['Goals_vs_xG'] > 3
else:
    df_filtered['Goals_vs_xG'] = 0
    df_filtered['Is_Overperformer'] = False

print("PLAYER DISTRIBUTION BY TIER:")
print(df_filtered['Elite_Tier'].value_counts().sort_index())
print(f"\nUndervalued Players: {df_filtered['Is_Undervalued'].sum()}")
print(f"Overperformers (xG): {df_filtered['Is_Overperformer'].sum()}")

## [5] CORRELATION & STATISTICAL ANALYSIS

In [ ]:
# Correlation matrix
corr_cols = ['Age', 'Goals', 'Assists', 'Minutes', 'Shots', 'Shot_Accuracy',
             'Goals_per90', 'Assists_per90', 'Performance_Score']
corr_cols = [c for c in corr_cols if c in df.columns]

correlation_matrix = df[corr_cols].corr()

print("TOP CORRELATIONS with Performance_Score:")
print(correlation_matrix['Performance_Score'].sort_values(ascending=False).head(6))

# Position analysis
print("\n" + "="*80)
print("PERFORMANCE BY POSITION:")
print("="*80)
position_stats = df_filtered.groupby('Position').agg({
    'Goals_per90': ['mean', 'count'],
    'Assists_per90': 'mean',
    'Performance_Score': ['mean', 'max'],
}).round(2)
display(position_stats)

## [6] ELITE PLAYERS IDENTIFICATION

In [ ]:
print("🏆 TOP 15 ELITE PLAYERS (by Performance Score)\n")
elite_players = df_filtered.nlargest(15, 'Performance_Score')[[
    'Player Name', 'Club', 'Position', 'Age', 'Goals', 'Assists', 
    'Minutes', 'Performance_Score', 'Elite_Tier'
]]
display(elite_players)

print("\n🎯 TOP PERFORMERS BY POSITION:\n")
for pos in df_filtered['Position'].unique():
    top_pos = df_filtered[df_filtered['Position'] == pos].nlargest(3, 'Performance_Score')
    print(f"\n{pos}:")
    for idx, player in top_pos.iterrows():
        print(f"  • {player['Player Name']} ({player['Club']}) - Score: {player['Performance_Score']:.2f}")

## [7] UNDERVALUED PLAYERS & EMERGING TALENT

In [ ]:
print("💎 UNDERVALUED GEMS (High performance, Low playing time)\n")
undervalued_df = df_filtered[df_filtered['Is_Undervalued']].nlargest(15, 'Performance_Score')[[
    'Player Name', 'Club', 'Position', 'Age', 'Performance_Score', 'Minutes', 
    'Goals_per90', 'Assists_per90'
]]
display(undervalued_df)

print("\n🌟 YOUNG TALENTS (Age < 23, Elite tier)\n")
young_talent = df_filtered[
    (df_filtered['Age'] < 23) & 
    (df_filtered['Elite_Tier'] == 'Elite')
].nlargest(10, 'Performance_Score')[[
    'Player Name', 'Club', 'Position', 'Age', 'Performance_Score', 'Goals_per90'
]]
display(young_talent)

## [8] CLUSTERING: PLAYER ARCHETYPES

In [ ]:
# Prepare clustering
clustering_features = ['Goals_per90', 'Assists_per90', 'Shot_Accuracy', 'Age']
X_cluster = df_filtered[clustering_features].fillna(0)

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# K-means (4 archetypes)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_filtered['Player_Archetype'] = kmeans.fit_predict(X_scaled)

# Label archetypes
archetype_names = {
    0: 'Defensive/Support',
    1: 'Creative Playmaker',
    2: 'Goal Scorer',
    3: 'Versatile Attacker'
}
df_filtered['Player_Archetype'] = df_filtered['Player_Archetype'].map(archetype_names)

print("✓ 4 Player Archetypes identified\n")
print("ARCHETYPE DISTRIBUTION:")
print(df_filtered['Player_Archetype'].value_counts())

print("\nARCHETYPE CHARACTERISTICS:")
archetype_stats = df_filtered.groupby('Player_Archetype')[clustering_features].mean().round(2)
display(archetype_stats)

print("\nTOP PLAYERS PER ARCHETYPE:")
for arch in df_filtered['Player_Archetype'].unique():
    top = df_filtered[df_filtered['Player_Archetype'] == arch].nlargest(2, 'Performance_Score')
    print(f"\n{arch}:")
    for _, player in top.iterrows():
        print(f"  • {player['Player Name']} ({player['Club']})")

## [9] CLUB PERFORMANCE ANALYSIS

In [ ]:
club_stats = df_filtered.groupby('Club').agg({
    'Player Name': 'count',
    'Goals_per90': 'mean',
    'Assists_per90': 'mean',
    'Performance_Score': 'mean',
    'Age': 'mean',
    'Minutes': 'sum'
}).round(2)
club_stats.columns = ['Players', 'Avg_Goals_per90', 'Avg_Assists_per90', 
                      'Avg_Performance', 'Avg_Age', 'Total_Minutes']
club_stats = club_stats.sort_values('Avg_Performance', ascending=False)

print("TOP 15 CLUBS BY AVERAGE PLAYER PERFORMANCE:\n")
display(club_stats.head(15))

## [10] INTERACTIVE VISUALIZATIONS (Plotly)

In [ ]:
# Chart 1: Performance vs Minutes
fig1 = px.scatter(
    df_filtered,
    x='Minutes', y='Performance_Score',
    color='Position', size='Goals_per90',
    hover_name='Player Name',
    hover_data={'Club': True, 'Age': True, 'Goals': True, 'Assists': True},
    title='<b>Player Performance vs Playing Time</b><br><sub>Size = Goals per 90</sub>',
    template='plotly_white'
)
fig1.update_layout(width=1200, height=700)
fig1.write_html('visualizations/01_performance_vs_minutes.html')
fig1.show()

In [ ]:
# Chart 2: Goals vs xG (if available)
if df_filtered['xG'].sum() > 0:
    fig2 = px.scatter(
        df_filtered[df_filtered['xG'] > 0],
        x='xG', y='Goals',
        color='Goals_vs_xG',
        size='Minutes',
        hover_name='Player Name',
        hover_data={'Club': True, 'Position': True},
        title='<b>Goals vs Expected Goals (xG)</b><br><sub>Who Over/Underperforms?</sub>',
        template='plotly_white',
        color_continuous_scale='RdYlGn'
    )
    # Add diagonal line
    fig2.add_shape(type="line", x0=0, y0=0, x1=30, y1=30, 
                   line=dict(color="gray", width=2, dash="dash"))
    fig2.update_layout(width=1200, height=700)
    fig2.write_html('visualizations/02_goals_vs_xg.html')
    fig2.show()

In [ ]:
# Chart 3: Position Heatmap
position_features = ['Goals_per90', 'Assists_per90', 'Shot_Accuracy', 
                     'Performance_Score', 'Cards_per90']
position_heatmap = df_filtered.groupby('Position')[position_features].mean()

fig3 = go.Figure(data=go.Heatmap(
    z=position_heatmap.values,
    x=position_heatmap.columns,
    y=position_heatmap.index,
    colorscale='Viridis',
    text=np.round(position_heatmap.values, 2),
    texttemplate='%{text:.2f}',
    textfont={"size": 11}
))
fig3.update_layout(
    title='<b>Position Performance Heatmap</b>',
    xaxis_title='Metrics',
    yaxis_title='Position',
    width=1000, height=600
)
fig3.write_html('visualizations/03_position_heatmap.html')
fig3.show()

In [ ]:
# Chart 4: Top 20 Players
top_20 = df_filtered.nlargest(20, 'Performance_Score')
fig4 = px.bar(
    top_20.sort_values('Performance_Score'),
    x='Performance_Score', y='Player Name',
    color='Position',
    hover_data={'Club': True, 'Age': True, 'Goals': True},
    title='<b>Top 20 Elite Players</b>',
    template='plotly_white'
)
fig4.update_layout(width=1000, height=700)
fig4.write_html('visualizations/04_top_20_players.html')
fig4.show()

In [ ]:
# Chart 5: Sunburst (Position → Club → Player)
fig5 = px.sunburst(
    df_filtered.nlargest(50, 'Performance_Score'),
    path=['Position', 'Club', 'Player Name'],
    values='Performance_Score',
    color='Performance_Score',
    color_continuous_scale='Greens',
    title='<b>Top 50 Players Hierarchy</b>'
)
fig5.update_layout(width=1000, height=900)
fig5.write_html('visualizations/05_player_hierarchy.html')
fig5.show()

In [ ]:
# Chart 6: Archetype Distribution
archetype_dist = df_filtered['Player_Archetype'].value_counts().reset_index()
archetype_dist.columns = ['Player_Archetype', 'Count']
fig6 = px.pie(
    archetype_dist,
    names='Player_Archetype', values='Count',
    title='<b>Player Archetype Distribution</b>',
    template='plotly_white'
)
fig6.update_traces(textposition='inside', textinfo='percent+label')
fig6.update_layout(width=800, height=600)
fig6.write_html('visualizations/06_archetype_distribution.html')
fig6.show()

In [ ]:
# Chart 7: Club Comparison
top_clubs = club_stats.head(15).reset_index()
fig7 = px.bar(
    top_clubs,
    x='Club', y='Avg_Performance',
    color='Avg_Goals_per90',
    title='<b>Top 15 Clubs by Average Player Performance</b>',
    template='plotly_white',
    color_continuous_scale='Blues'
)
fig7.update_layout(xaxis_tickangle=-45, width=1200, height=600)
fig7.write_html('visualizations/07_club_comparison.html')
fig7.show()

In [ ]:
# Chart 8: Age vs Performance
fig8 = px.scatter(
    df_filtered,
    x='Age', y='Performance_Score',
    color='Position', size='Minutes',
    hover_name='Player Name',
    hover_data={'Club': True, 'Goals_per90': ':.2f'},
    title='<b>Age vs Performance</b><br><sub>Size = Minutes Played</sub>',
    template='plotly_white'
)
fig8.update_layout(width=1200, height=700)
fig8.write_html('visualizations/08_age_vs_performance.html')
fig8.show()

In [ ]:
# Chart 9: Undervalued Players
if df_filtered['Is_Undervalued'].sum() > 0:
    undervalued_top = df_filtered[df_filtered['Is_Undervalued']].nlargest(15, 'Performance_Score')
    fig9 = px.scatter(
        undervalued_top,
        x='Minutes', y='Performance_Score',
        color='Position', size='Goals_per90',
        hover_name='Player Name',
        hover_data={'Club': True, 'Age': True},
        title='<b>Undervalued Players</b><br><sub>High Performance, Low Minutes</sub>',
        template='plotly_white'
    )
    fig9.update_layout(width=1000, height=700)
    fig9.write_html('visualizations/09_undervalued_players.html')
    fig9.show()

## [11] DATA EXPORTS FOR POWER BI

In [ ]:
print("[EXPORTING DATA FOR POWER BI]\n")

# Export 1: Full player data
export_full = df_filtered[[
    'Player Name', 'Club', 'Position', 'Age', 'Age_Group',
    'Goals', 'Assists', 'Minutes', 'Appearances',
    'Goals_per90', 'Assists_per90', 'Shot_Accuracy', 'Conversion_Rate',
    'Performance_Score', 'Elite_Tier', 'Player_Archetype',
    'Is_Undervalued', 'Is_Overperformer', 'Goals_vs_xG',
    'Total_Cards', 'Cards_per90'
]].copy()
export_full = export_full.sort_values('Performance_Score', ascending=False)
export_full.to_csv('exports/01_players_full_stats.csv', index=False, encoding='utf-8-sig')
print("✓ 01_players_full_stats.csv")

# Export 2: Elite players
scout_report = export_full[export_full['Elite_Tier'] == 'Elite'].head(50)
scout_report.to_csv('exports/02_scout_report_elite.csv', index=False, encoding='utf-8-sig')
print("✓ 02_scout_report_elite.csv")

# Export 3: Undervalued
undervalued_export = export_full[export_full['Is_Undervalued']].head(30)
undervalued_export.to_csv('exports/03_undervalued_gems.csv', index=False, encoding='utf-8-sig')
print("✓ 03_undervalued_gems.csv")

# Export 4: Young talents
young_talents_export = export_full[(export_full['Age'] < 23) & (export_full['Elite_Tier'] == 'Elite')]
young_talents_export.to_csv('exports/04_young_talents.csv', index=False, encoding='utf-8-sig')
print("✓ 04_young_talents.csv")

# Export 5: Club summary
club_export = club_stats.reset_index()
club_export.to_csv('exports/05_club_summary.csv', index=False, encoding='utf-8-sig')
print("✓ 05_club_summary.csv")

# Export 6: Position analysis
position_export = df_filtered.groupby('Position').agg({
    'Player Name': 'count',
    'Goals_per90': ['mean', 'median', 'max'],
    'Assists_per90': ['mean', 'median', 'max'],
    'Performance_Score': ['mean', 'max'],
    'Age': 'mean'
}).reset_index()
position_export.to_csv('exports/06_position_analysis.csv', index=False, encoding='utf-8-sig')
print("✓ 06_position_analysis.csv")

print("\n✅ All CSV exports ready for Power BI!")

## [12] SUMMARY & KEY INSIGHTS

In [ ]:
print("="*80)
print("📋 SCOUT REPORT SUMMARY - KEY INSIGHTS")
print("="*80)

print("\n🏆 ELITE INSIGHTS:")
print(f"  • Total Elite Players: {(export_full['Elite_Tier'] == 'Elite').sum()}")
print(f"  • Top Performer: {export_full.iloc[0]['Player Name']} ({export_full.iloc[0]['Club']})")
print(f"    Score: {export_full.iloc[0]['Performance_Score']:.2f}")
print(f"  • Highest Goals/90: {df_filtered.loc[df_filtered['Goals_per90'].idxmax(), 'Player Name']} "
      f"({df_filtered['Goals_per90'].max():.2f})")
print(f"  • Highest Assists/90: {df_filtered.loc[df_filtered['Assists_per90'].idxmax(), 'Player Name']} "
      f"({df_filtered['Assists_per90'].max():.2f})")

print("\n💎 RECRUITMENT OPPORTUNITIES:")
print(f"  • Undervalued Players: {df_filtered['Is_Undervalued'].sum()}")
print(f"  • Young Talents (U-23): {((df_filtered['Age'] < 23) & (df_filtered['Elite_Tier'] == 'Elite')).sum()}")
if df_filtered['Is_Overperformer'].sum() > 0:
    print(f"  • Overperformers (vs xG): {df_filtered['Is_Overperformer'].sum()}")

print("\n📊 POSITION BREAKDOWN:")
for pos in sorted(df_filtered['Position'].unique()):
    pos_data = df_filtered[df_filtered['Position'] == pos]
    print(f"  • {pos}: {len(pos_data)} players | Avg Score: {pos_data['Performance_Score'].mean():.2f}")

print("\n🔗 TOP CLUBS:")
top_3_clubs = club_stats.head(3)
for i, club in enumerate(top_3_clubs.index, 1):
    print(f"  {i}. {club}: {top_3_clubs.loc[club, 'Avg_Performance']:.2f} avg performance")

print("\n" + "="*80)
print("✅ ANALYSIS COMPLETE - All exports ready for Power BI Dashboard")
print("="*80)